```
------------- ESPECIALIZACIÓN EN INTELIGENCIA ARTIFICIAL Y BIG DATA -------------
---------------------------------------------------------------------------------

Módulo:                     BIG DATA APLICADO
Profesor:                   Víctor J. González
Unidad de Trabajo:          UT03. Data Lakes y almacenamiento distribuido en cloud con S3
Práctica:                   PR0301. Ingesta en Data Lake (Capa Bronce) sobre Amazon S3
Resultados de aprendizaje:  RA1
```



# PR0301. Ingesta en Data Lake (Capa Bronce) sobre Amazon S3

## 1. Contexto

La Secretaría de Estado de Turismo está desarrollando un sistema de inteligencia de datos para monitorizar el estado y la ocupación de las playas españolas. Como Ingeniero de Datos, tu primer objetivo es construir el pipeline de ingesta hacia la **Capa Bronce (Raw Data)** de un Data Lake en **Amazon S3**.

Al analizar las fuentes oficiales, te has encontrado con un problema habitual en proyectos reales de datos: **el Ministerio para la Transición Ecológica (MITECO) y la Agencia Estatal de Meteorología (AEMET) utilizan identificadores distintos para las mismas playas**.

Por tanto, tu misión en esta Capa Bronce consiste en centralizar en su formato original (**sin transformar ni cruzar aún las tablas**) las tres piezas de información necesarias para que, en fases posteriores (Capa Plata), los analistas puedan unificarlas:

1. **Catálogo de Infraestructura de Playas (MITECO):** dataset detallado con características físicas, servicios y accesibilidad de más de 3.000 playas.
2. **Catálogo Maestro de Códigos de AEMET:** diccionario oficial de AEMET que asigna el identificador meteorológico (`ID_PLAYA`) a las playas de cada municipio costero.
3. **Predicción Meteorológica Diaria (API REST AEMET):** datos diarios en tiempo real de estado del mar, viento, oleaje e índice UV obtenidos mediante el código `ID_PLAYA`.

Trabajarás siguiendo el flujo estándar de la industria: primero realizarás la **exploración y prototipado en Jupyter Notebook** y, una vez validada la lógica, **refactorizarás el código a una arquitectura modular de scripts Python (`.py`)** lista para producción.





## 2. Objetivos de aprendizaje

Los objetivos de esta práctica son:

- **Arquitectura de Data Lake (Capa Bronce):** implementar esquemas de versionado para múltiples catálogos maestros de distintos orígenes y particionamiento temporal estilo Hive (`year=YYYY/month=MM/day=DD`) para series temporales en Amazon S3.
- **Interoperabilidad entre fuentes heterogéneas:** manejar archivos CSV con distintas codificaciones (`UTF-8` vs `ISO-8859-1`/`cp1252`), distintos delimitadores (`,` vs `;`) y relacionar fuentes dispares a través de códigos estándar (Código INE de municipio).
- **Consumo avanzado de APIs REST:** gestionar el flujo de doble petición de AEMET OpenData, controlando límites de tasa (*Rate Limiting*) y códigos de estado HTTP.
- **SDK de AWS (`boto3`) y paso a producción:** realizar subidas desde disco y desde memoria RAM (`put_object`) con metadatos nativos en S3, empaquetando el código de un Notebook en scripts modulares parametrizados con `.env`.





## 3. Fuentes de datos y análisis de esquemas

### A. Catálogo de infraestructura de playas — MITECO (`./data/playas.csv`)

[Fichero CSV](./data/playas.csv)

Extraído del [Portal OpenData de ESRI / MITECO](https://opendata.esri.es/datasets/84ddbc8cf4104a579d579f6441fcaa8a_0).

Es un fichero en formato CSV separado por comas (`,`), codificado en `UTF-8`. Los campos clave de este fichero son:

- `Identifica`: identificador interno del MITECO (ej. `316` para La Venus). Observa que este es un campo interno de MITECO y no sirve para consultar la API de AEMET.
- `Código_IN`: código oficial INE del municipio de 5 dígitos (ej. `29069` para Marbella). Este campo es el que se utilizaría al realizar el paso a la capa plata como nexo de unión con el catálogo de AEMET.
- `Nombre`, `Provincia`, `Término_M` (Municipio).
- Atributos de negocio (para futura Capa Plata): `Duchas`, `Aseos`, `Acceso_dis`, `Bandera_az`, `Grado_ocup`, `Grado_urba`, `Coordena_4` (Longitud decimal), `Coordena_5` (Latitud decimal).



### B. Diccionario de playas de AEMET (`./data/playas_codigos.csv`)

[Fichero CSV](./data/Playas_codigos.csv)

Fichero maestro [publicado por AEMET](https://www.aemet.es/documentos/es/eltiempo/prediccion/playas/Playas_codigos.csv) con las playas que disponen de predicción meteorológica.

El formato de este fichero es un CSV separado por punto y coma (`;`), codificado en `ISO-8859-1` (`latin1` o `cp1252`). Si lo abres en UTF-8 por defecto, verás caracteres corruptos en nombres con tildes o caracteres especiales como `l'Alfàs del Pi`.

Sus campos clave son:

- `ID_PLAYA`: código de 7 dígitos requerido por la API de AEMET (ej. `0301101`). 
- `ID_MUNICIPIO`: código INE del municipio de 5 dígitos (ej. `03011`), que permite cruzar los municipios de este fichero con la columna `Código_IN` de `playas.csv`.
- `NOMBRE_PLAYA`, `ID_PROVINCIA`, `NOMBRE_PROVINCIA`, `NOMBRE_MUNICIPIO`, `LATITUD`, `LONGITUD`.


### C. API AEMET OpenData (API REST)

Endpoint específico de predicción diaria de playas:

```text
GET https://opendata.aemet.es/opendata/api/prediccion/especifica/playa/{ID_PLAYA}
```

Ten en cuenta que para realizar consultas a la API de AEMET necesitas autenticación pasando tu API Key en la cabecera `api_key`. Además, la consulta se realiza mediante un flujo de doble salto:

1. La primera petición al endpoint (pasando el `ID_PLAYA` de 7 dígitos extraído de `playas_codigos.csv`) devuelve un JSON de control con un campo `datos` (URL temporal).
2. Debes realizar una segunda petición `GET` a esa URL temporal para descargar el JSON final con la predicción.

Por otro lado, AEMET tiene una restricción de volumen o cuota API estableciendo un límite de peticiones por minuto (`HTTP 429`), **selecciona una única provincia costera en `playas.csv**`, obtén los códigos INE de sus municipios (`Código_IN`), busca en `playas_codigos.csv` qué playas de AEMET pertenecen a esos municipios y **descarga la predicción para un máximo de 15 playas**, aplicando una pausa (`time.sleep(1.5)`) entre llamadas.




## 4. Especificaciones técnicas

### Fase 1: Diseño de la Capa Bronce en S3

En la Capa Bronce los datos se almacenan **en crudo (Raw)**, respetando el formato original de cada proveedor. Crea mediante código (`boto3`) un bucket en S3 llamado `datalake-turismo-bronce-<tus-iniciales>` (en minúsculas) con la siguiente jerarquía de claves (*S3 Keys*):

```text
s3://datalake-turismo-bronce-<iniciales>/
├── bronce/
│   ├── catalogos/
│   │   ├── miteco_playas/
│   │   │   └── v1/
│   │   │       └── playas.csv
│   │   └── aemet_codigos/
│   │       └── v1/
│   │           └── playas_codigos.csv
│   └── meteorologia/
│       └── aemet_prediccion_playas/
│           └── year=YYYY/
│               └── month=MM/
│                   └── day=DD/
│                       ├── pred_playa_<ID_PLAYA>_<HHMMSS>.json
│                       └── ...

```

Los dos catálogos maestros (`miteco_playas` y `aemet_codigos`) son estáticos y de proveedores distintos, por lo que se separan por fuente y se organizan por **versión (`v1/`)** para mantener la trazabilidad si alguno de los organismos modifica su estructura en el futuro. En cambio, las predicciones meteorológicas son eventos diarios y se almacenan con **particionamiento temporal (`year=.../month=.../day=...`)** para que motores analíticos posteriores (AWS Athena, Apache Spark) puedan filtrar por fechas leyendo solo las subcarpetas necesarias (*Partition Pruning*).





### Fase 2: Prototipado y exploración en Jupyter Notebook

Utiliza un cuaderno de Jupyter como laboratorio para resolver los retos técnicos antes de empaquetar el código:



In [1]:
%pip install boto3 requests pandas python-dotenv


Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import os
import time
from datetime import datetime, timezone
from pathlib import Path

import boto3
import pandas as pd
import requests
from dotenv import load_dotenv

PROJECT_ROOT = Path("..")
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = Path(".")
DATA_DIR = PROJECT_ROOT / "data"

load_dotenv(PROJECT_ROOT / ".env")

PROVINCIA = "Asturias"     
MAX_PLAYAS = 15            
BUCKET_NAME = os.getenv("S3_BUCKET_NAME", "datalake-turismo-bronce-plf")
AWS_REGION = os.getenv("AWS_DEFAULT_REGION", "us-east-1")
AEMET_API_KEY = os.getenv("AEMET_API_KEY")

print(f"DATA_DIR: {DATA_DIR.resolve()}")
for nombre in ("playas.csv", "playas_codigos.csv"):
    ruta = DATA_DIR / nombre
    print(f"  {nombre}: {'OK' if ruta.exists() else 'NO ENCONTRADO'}")

print(f"Bucket S3: {BUCKET_NAME} | Región: {AWS_REGION}")
print(f"AEMET_API_KEY: {'configurada' if AEMET_API_KEY else 'NO configurada (rellena .env)'}")


DATA_DIR: C:\Users\pabli\OneDrive\Escritorio\Inteligencia Artificial y Big Data V2\BigDataAplicado\bda\ut03\pr0301\data
  playas.csv: OK
  playas_codigos.csv: OK
Bucket S3: datalake-turismo-bronce-plf | Región: us-east-1
AEMET_API_KEY: configurada


1. **Inspección de los dos CSV y cruce por municipio (Código INE):** carga `playas.csv` y `playas_codigos.csv` comprobando el separador y el sistema de codificación.


In [3]:
df_miteco = pd.read_csv(
    DATA_DIR / "playas.csv",
    sep=",",
    encoding="utf-8-sig",
    dtype={"Código_IN": str},
)

df_aemet = pd.read_csv(
    DATA_DIR / "playas_codigos.csv",
    sep=";",
    encoding="cp1252",
    dtype={"ID_PLAYA": str, "ID_MUNICIPIO": str},
)

# Conservamos los ceros a la izquierda de los códigos oficiales
df_miteco["Código_IN"] = df_miteco["Código_IN"].str.zfill(5)
df_aemet["ID_PLAYA"] = df_aemet["ID_PLAYA"].str.zfill(7)
df_aemet["ID_MUNICIPIO"] = df_aemet["ID_MUNICIPIO"].str.zfill(5)

print("MITECO:", df_miteco.shape, "| sep=',' | encoding='utf-8-sig'")
print("AEMET :", df_aemet.shape, "| sep=';' | encoding='cp1252'")

columnas_clave_miteco = [
    "Identifica", "Código_IN", "Nombre", "Provincia", "Término_M",
    "Duchas", "Aseos", "Acceso_dis", "Bandera_az", "Grado_ocup", "Grado_urba",
    "Coordena_4", "Coordena_5",
]
print("\nMITECO (columnas clave):")
display(df_miteco[columnas_clave_miteco].head())

print("\nAEMET (cabecera):")
display(df_aemet.head())


MITECO: (3554, 80) | sep=',' | encoding='utf-8-sig'
AEMET : (591, 8) | sep=';' | encoding='cp1252'

MITECO (columnas clave):


,Identifica,Código_IN,Nombre,Provincia,Término_M,Duchas,Aseos,Acceso_dis,Bandera_az,Grado_ocup,Grado_urba,Coordena_4,Coordena_5
0,316.0,29069,La Venus,Málaga,Marbella,Sí,Sí,Sí,No,Alto,Urbana,-4.8867,36.5066
1,318.0,29069,Las Cañas,Málaga,Marbella,Sí,Sí,No,No,Medio,Urbana,-4.7601,36.4865
2,330.0,29070,Calahonda,Málaga,Mijas,Sí,Sí,Sí,Sí,Medio / Alto,Urbana,-4.7112,36.4877
3,402904.0,29070,El Charcón,Málaga,Mijas,Sí,No,No,No,Bajo,Semiurbana,-4.6456,36.5072
4,450301.0,03018,L'Espigó,Alicante/Alacant,Altea,No,Sí,Sí,Sí,Alto,Urbana,-0.0447,38.6024



AEMET (cabecera):


,ID_PLAYA,NOMBRE_PLAYA,ID_PROVINCIA,NOMBRE_PROVINCIA,ID_MUNICIPIO,NOMBRE_MUNICIPIO,LATITUD,LONGITUD
0,0301101,Raco de l'Albir,3,Alacant/Alicante,03011,l'Alfàs del Pi,"38º 34' 31""","-00º 03' 52"""
1,0301401,Sant Joan / San Juan,3,Alacant/Alicante,03014,Alicante/Alacant,"38º 22' 48""","-00º 24' 32"""
2,0301408,El Postiguet,3,Alacant/Alicante,03014,Alicante/Alacant,"38º 20' 46""","-00º 28' 38"""
3,0301410,Saladar,3,Alacant/Alicante,03014,Alicante/Alacant,"38º 17' 02""","-00º 31' 08"""
4,0301808,La Roda,3,Alacant/Alicante,03018,Altea,"38º 36' 29""","-00º 02' 16"""


In [4]:
es_provincia = df_miteco["Provincia"].astype(str).str.strip().str.upper() == PROVINCIA.upper()
df_provincia = df_miteco.loc[es_provincia].copy()

codigos_ine = sorted(df_provincia["Código_IN"].dropna().unique())

todos_aemet_provincia = df_aemet[df_aemet["ID_MUNICIPIO"].isin(codigos_ine)].copy()

playas_seleccionadas = (
    todos_aemet_provincia.sort_values("ID_PLAYA").head(MAX_PLAYAS).reset_index(drop=True)
)
ids_playa = playas_seleccionadas["ID_PLAYA"].tolist()

print(f"Provincia seleccionada: {PROVINCIA}")
print(f"Playas MITECO en {PROVINCIA}: {len(df_provincia)}")
print(f"Municipios INE únicos: {len(codigos_ine)} -> {codigos_ine}")
print(f"Playas AEMET para esos municipios: {len(todos_aemet_provincia)}")
print(f"Seleccionadas (máx. {MAX_PLAYAS}): {len(ids_playa)} -> {ids_playa}")

display(
    playas_seleccionadas[
        ["ID_PLAYA", "ID_MUNICIPIO", "NOMBRE_PLAYA", "NOMBRE_MUNICIPIO", "NOMBRE_PROVINCIA"]
    ]
)


Provincia seleccionada: Asturias
Playas MITECO en Asturias: 208
Municipios INE únicos: 19 -> ['33013', '33014', '33016', '33017', '33018', '33019', '33021', '33023', '33024', '33025', '33034', '33036', '33039', '33041', '33055', '33056', '33069', '33070', '33076']
Playas AEMET para esos municipios: 26
Seleccionadas (máx. 15): 15 -> ['3301303', '3301403', '3301406', '3301608', '3301704', '3301804', '3301903', '3302117', '3302120', '3302302', '3302403', '3302509', '3303402', '3303407', '3303612']


,ID_PLAYA,ID_MUNICIPIO,NOMBRE_PLAYA,NOMBRE_MUNICIPIO,NOMBRE_PROVINCIA
0,3301303,33013,Arenal de Moris,Caravia,Asturias
1,3301403,33014,Candás - La Palmera,Carreño,Asturias
2,3301406,33014,Xivares,Carreño,Asturias
3,3301608,33016,"Salinas - San Juan, El Espartal",Castrillón,Asturias
4,3301704,33017,Peñarronda,Castropol,Asturias
5,3301804,33018,Arnielles,Coaña,Asturias
6,3301903,33019,La Isla,Colunga,Asturias
7,3302117,33021,San Pedro de Bocamar,Cudillero,Asturias
8,3302120,33021,La Concha de Artedo,Cudillero,Asturias
9,3302302,33023,Pormenande,El Franco,Asturias


2. **Prueba unitaria contra la API de AEMET:** toma un `ID_PLAYA` válido (ej. `"0301101"` o uno de tu provincia elegida), realiza la petición de doble salto a AEMET y verifica que el JSON descargado muestra correctamente las tildes y las eñes.


In [5]:
def obtener_prediccion_aemet(id_playa, api_key):
    url = f"https://opendata.aemet.es/opendata/api/prediccion/especifica/playa/{id_playa}"
    headers = {"api_key": api_key, "Accept": "application/json"}
    try:
        resp_control = requests.get(url, headers=headers, timeout=10)
        if resp_control.status_code == 429:
            print(f"[429] Límite de peticiones alcanzado para {id_playa}")
            return None
        resp_control.raise_for_status()
        info_control = resp_control.json()

        if info_control.get("estado") != 200:
            print(
                f"[{info_control.get('estado')}] Sin predicción para {id_playa}: "
                f"{info_control.get('descripcion')}"
            )
            return None

        resp_datos = requests.get(info_control["datos"], timeout=10)
        if resp_datos.status_code == 429:
            print(f"[429] Límite de peticiones al descargar datos de {id_playa}")
            return None
        resp_datos.raise_for_status()
        resp_datos.encoding = "ISO-8859-15"
        return resp_datos.json()

    except requests.RequestException as error:
        print(f"Error de red/API para {id_playa}: {error}")
        return None


datos_prediccion = None
id_prueba = ids_playa[0] if ids_playa else None

if not AEMET_API_KEY or id_prueba is None:
    print("Falta AEMET_API_KEY o no hay ID_PLAYA seleccionado; no se llama a la API.")
else:
    print(f"Probando con ID_PLAYA={id_prueba}")
    datos_prediccion = obtener_prediccion_aemet(id_prueba, AEMET_API_KEY)
    if datos_prediccion is not None:
        print("JSON final (primeros 1500 caracteres, con tildes y eñes):")
        print(json.dumps(datos_prediccion, ensure_ascii=False, indent=2)[:1500])
    # Pausa para respetar el rate limiting de AEMET
    time.sleep(1.5)


Probando con ID_PLAYA=3301303
JSON final (primeros 1500 caracteres, con tildes y eñes):
[
  {
    "origen": {
      "productor": "Agencia Estatal de Meteorología - AEMET. Gobierno de España",
      "web": "http://www.aemet.es",
      "language": "es",
      "copyright": "© AEMET. Autorizado el uso de la información y su reproducción citando a AEMET como autora de la misma.",
      "notaLegal": "http://www.aemet.es/es/nota_legal"
    },
    "elaborado": "2026-11-24T08:51:20",
    "nombre": "Arenal de Moris",
    "localidad": -32523,
    "prediccion": {
      "dia": [
        {
          "estadoCielo": {
            "value": "",
            "f1": 110,
            "descripcion1": "nuboso",
            "f2": 140,
            "descripcion2": "muy nuboso con lluvia"
          },
          "viento": {
            "value": "",
            "f1": 210,
            "descripcion1": "flojo",
            "f2": 210,
            "descripcion2": "flojo"
          },
          "oleaje": {
            "va

3. **Prueba de subida a S3 con `boto3`:**  carga tus credenciales desde `.env`, conecta con S3 y prueba tanto la subida de archivos locales (`upload_file`) como la subida de un JSON en memoria (`put_object`) adjuntando metadatos personalizados (`Metadata={...}`).





In [6]:

s3_client = boto3.client("s3")

try:
    s3_client.head_bucket(Bucket=BUCKET_NAME)
    print(f"El bucket {BUCKET_NAME} ya existe.")
except s3_client.exceptions.ClientError:
    kwargs = {"Bucket": BUCKET_NAME}
    if AWS_REGION != "us-east-1":
        kwargs["CreateBucketConfiguration"] = {"LocationConstraint": AWS_REGION}
    s3_client.create_bucket(**kwargs)
    print(f"Bucket creado: {BUCKET_NAME}")

ahora = datetime.now(timezone.utc)
timestamp_iso = ahora.strftime("%Y-%m-%dT%H:%M:%SZ")
ruta_particion = f"year={ahora.year}/month={ahora.month:02d}/day={ahora.day:02d}"

print(f"ingestion_timestamp: {timestamp_iso}")
print(f"Partición: {ruta_particion}")


El bucket datalake-turismo-bronce-plf ya existe.
ingestion_timestamp: 2026-11-24T16:54:58Z
Partición: year=2026/month=11/day=24


In [7]:
key_miteco = "bronce/catalogos/miteco_playas/v1/playas.csv"
key_aemet_codigos = "bronce/catalogos/aemet_codigos/v1/playas_codigos.csv"

s3_client.upload_file(
    Filename=str(DATA_DIR / "playas.csv"),
    Bucket=BUCKET_NAME,
    Key=key_miteco,
    ExtraArgs={
        "Metadata": {"source": "MITECO_OpenData", "ingestion_timestamp": timestamp_iso}
    },
)
print(f"Subido: s3://{BUCKET_NAME}/{key_miteco}")

s3_client.upload_file(
    Filename=str(DATA_DIR / "playas_codigos.csv"),
    Bucket=BUCKET_NAME,
    Key=key_aemet_codigos,
    ExtraArgs={
        "Metadata": {"source": "AEMET_Catalogo", "ingestion_timestamp": timestamp_iso}
    },
)
print(f"Subido: s3://{BUCKET_NAME}/{key_aemet_codigos}")

metadatos_miteco = s3_client.head_object(Bucket=BUCKET_NAME, Key=key_miteco)["Metadata"]
print("Metadatos comprobados en S3:", metadatos_miteco)


Subido: s3://datalake-turismo-bronce-plf/bronce/catalogos/miteco_playas/v1/playas.csv
Subido: s3://datalake-turismo-bronce-plf/bronce/catalogos/aemet_codigos/v1/playas_codigos.csv
Metadatos comprobados en S3: {'ingestion_timestamp': '2026-11-24T16:54:58Z', 'source': 'MITECO_OpenData'}


In [8]:
key_json = None

if datos_prediccion is None:
    print("No hay JSON de AEMET; se omite el put_object.")
else:
    json_en_bytes = json.dumps(datos_prediccion, ensure_ascii=False, indent=2).encode("utf-8")
    key_json = (
        f"bronce/meteorologia/aemet_prediccion_playas/{ruta_particion}/"
        f"pred_playa_{id_prueba}_{ahora.strftime('%H%M%S')}.json"
    )
    s3_client.put_object(
        Bucket=BUCKET_NAME,
        Key=key_json,
        Body=json_en_bytes,
        ContentType="application/json",
        Metadata={"source": "AEMET_API", "ingestion_timestamp": timestamp_iso},
    )
    print(f"Subido desde RAM: s3://{BUCKET_NAME}/{key_json} ({len(json_en_bytes)} bytes)")


Subido desde RAM: s3://datalake-turismo-bronce-plf/bronce/meteorologia/aemet_prediccion_playas/year=2026/month=11/day=24/pred_playa_3301303_165458.json (4391 bytes)


In [9]:
respuesta = s3_client.list_objects_v2(Bucket=BUCKET_NAME, Prefix="bronce/")
objetos = respuesta.get("Contents", [])

print(f"Objetos bajo bronce/: {len(objetos)}")
for objeto in objetos:
    print(f"  s3://{BUCKET_NAME}/{objeto['Key']} ({objeto['Size']} bytes)")


Objetos bajo bronce/: 21
  s3://datalake-turismo-bronce-plf/bronce/catalogos/aemet_codigos/v1/playas_codigos.csv (45445 bytes)
  s3://datalake-turismo-bronce-plf/bronce/catalogos/miteco_playas/v1/playas.csv (3479191 bytes)
  s3://datalake-turismo-bronce-plf/bronce/meteorologia/aemet_prediccion_playas/year=2026/month=11/day=24/pred_playa_3301303_101747.json (4391 bytes)
  s3://datalake-turismo-bronce-plf/bronce/meteorologia/aemet_prediccion_playas/year=2026/month=11/day=24/pred_playa_3301303_104859.json (4391 bytes)
  s3://datalake-turismo-bronce-plf/bronce/meteorologia/aemet_prediccion_playas/year=2026/month=11/day=24/pred_playa_3301303_164220.json (4391 bytes)
  s3://datalake-turismo-bronce-plf/bronce/meteorologia/aemet_prediccion_playas/year=2026/month=11/day=24/pred_playa_3301303_164435.json (4391 bytes)
  s3://datalake-turismo-bronce-plf/bronce/meteorologia/aemet_prediccion_playas/year=2026/month=11/day=24/pred_playa_3301303_165458.json (4391 bytes)
  s3://datalake-turismo-bronce-p

### Fase 3: Paso a Producción (refactorización a scripts modulares)

Una vez validada la lógica en Jupyter, **empaqueta y refactoriza tu código** en scripts `.py` dentro de la carpeta `src/` para que el pipeline pueda ejecutarse de forma desatendida en un servidor o contenedor:

```text
pr0301_ingesta_bronce/
├── data/
│   ├── playas.csv                     # Catálogo MITECO
│   └── playas_codigos.csv             # Diccionario de códigos AEMET
├── notebooks/
│   └── 01_exploracion_ingesta.ipynb   # Fase 2: Laboratorio y exploración
├── src/                               # Fase 3: Código de Producción
│   ├── config.py                      # Carga de .env y constantes globales
│   ├── s3_manager.py                  # Funciones de subida y metadatos con boto3
│   ├── aemet_client.py                # Cliente de AEMET
│   └── main.py                        # Orquestador principal del pipeline
├── .env.example                       # Plantilla de variables (SIN claves reales)
├── .gitignore                         # Debe excluir .env, __pycache__/ y .ipynb_checkpoints/
└── requirements.txt                   # boto3, requests, pandas, python-dotenv

```

#### Requisitos de los módulos en `src/`:

1. `config.py`: este código se deberá de encargar de la configuración y seguridad.Cargará desde `.env` las variables `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`, `AWS_SESSION_TOKEN`, `AWS_DEFAULT_REGION`, `S3_BUCKET_NAME` y `AEMET_API_KEY`.
2. `s3_manager.py`: el cometido de este fichero será la gestión del Data Lake y la trazabilidad. Deberá contener una función para subir los ficheros desde disco y otra para volcar los datos en memoria. Todo objeto subido a S3 debe incluir **metadatos nativos de S3** (`Metadata={...}`) con los siguientes valores:
    - `source`: `"MITECO_OpenData"` (para `playas.csv`), `"AEMET_Catalogo"` (para `playas_codigos.csv`) o `"AEMET_API"` (para los JSON diarios).
    - `ingestion_timestamp`: fecha y hora exacta de subida en formato ISO 8601 UTC (`YYYY-MM-DDTHH:MM:SSZ`).
3. `aemet_client.py`: aquí incluirás el código de extracción de datos de la API de AEMET. Tendrá una función que recibe un `id_playa` (7 dígitos) y realiza el doble salto contra AEMET devolviendo el JSON final. Lo ideal es que gestione posibles errores HTTP (`404`, `429`, timeouts) registrando un `WARNING` y devolviendo `None` sin detener la ejecución general.
4. `main.py`: este fichero será el orquestador del pipeline y es el que se ejecutará desde el terminal mediante `python src/main.py`. La secuencia de ejecución es:
     1. Sube los dos ficheros `csv` a sus respectivas rutas `v1/` en S3.
     2. Lee `playas.csv`, filtra una provincia costera y obtiene el listado único de códigos INE de sus municipios (`Código_IN`).
     3. Filtra `playas_codigos.csv` quedándose con las playas cuyos `ID_MUNICIPIO` estén en esa lista (máximo 15 playas) y extrae sus `ID_PLAYA` (7 dígitos).
     4. Descarga la predicción de cada `ID_PLAYA` y la sube directamente desde memoria RAM a S3 bajo la partición de la fecha actual (`year=YYYY/month=MM/day=DD/pred_playa_<ID_PLAYA>_<HHMMSS>.json`).
     5. Registra el progreso por consola usando `logging` y muestra un resumen final de archivos subidos.





## 5. Entregables

Esta práctica debes entregarla en el repositorio en un documento Markdown siguiendo el formato habitual y con lo siguiente:
    - El cuaderno de Jupyter ejecutado que has utilizado exportado a Markdown. 
    - Capturas de pantalla del resultado de ejecutar el código de producción.
    - Toda la estructura de directorios comprimida en un fichero ZIP



